# 🚀 Training & Fine-Tuning YOLO P&ID (Google Colab Free GPU T4)
Notebook ini dibuat khusus untuk **Tim IT** guna melatih ulang model YOLO deteksi aset P&ID.

### 📋 Prasyarat Sebelum Mulai:
1. Di menu atas, pastikan runtime menggunakan GPU: **Runtime** -> **Change runtime type** -> **T4 GPU** -> **Save**.
2. Upload file `yolo_dataset.zip` hasil ekspor dari server aplikasi ke panel **Files** (ikon folder di sebelah kiri).

In [ ]:
# 1. Periksa Akses GPU NVIDIA T4
!nvidia-smi

In [ ]:
# 2. Install Library Ultralytics & ONNX (~30 detik)
!pip install -q ultralytics onnx onnxruntime onnxslim

In [ ]:
# 3. Ekstrak File Dataset yang Diupload
import os
import shutil

if os.path.exists('yolo_dataset.zip'):
    !unzip -q -o yolo_dataset.zip -d dataset/
    print("✅ Dataset berhasil diekstrak ke folder 'dataset/'!")
else:
    raise FileNotFoundError("❌ File 'yolo_dataset.zip' tidak ditemukan. Silakan upload file zip ke panel Files terlebih dahulu!")

In [ ]:
# 4. Jalankan Pelatihan / Fine-Tuning (Estimasi: 15-20 Menit)
from ultralytics import YOLO

print("🚀 Memulai Fine-Tuning YOLOv8 Nano...")
model = YOLO('yolov8n.pt')  # Menggunakan pre-trained Nano model

results = model.train(
    data='dataset/data.yaml',
    epochs=50,
    imgsz=1024,
    batch=8,
    device=0,
    optimizer='AdamW',
    lr0=0.001,
    name='pid_finetune',
    save=True,
    verbose=True
)
print("🎉 Training Selesai!")

In [ ]:
# 5. Ekspor Model Terbaik ke Format ONNX (Untuk Server Aplikasi)
best_checkpoint = 'runs/detect/pid_finetune/weights/best.pt'
trained_model = YOLO(best_checkpoint)

print("📦 Mengekspor model ke ONNX...")
onnx_file = trained_model.export(
    format='onnx',
    imgsz=1024,
    opset=17,
    simplify=True
)

# Beri nama sesuai model di server
final_name = 'yolo-pid-v8n-203.onnx'
shutil.copy(onnx_file, final_name)

print(f"🎉 SELESAI! File '{final_name}' siap didownload.")
print("👉 Klik kanan file 'yolo-pid-v8n-203.onnx' di panel Files sebelah kiri, lalu pilih Download.")